# 26-Class ECG Classifier V3 (PTB-XL + Chapman + Challenge 2021)

Trains ECGNetJoint on **~111K records** (18,524 PTB-XL + 42,390 Chapman + ~50K Challenge).
Transfer learning from 14-class v2 model. Adds 12 new conditions.

---
## Runtime guide

| Cell | Runtime | Duration | What it does |
|------|---------|----------|--------------|
| 1 | **CPU** | 1 min | Mount Drive + install deps |
| 2 | **CPU** | 1 min | Copy scripts from Drive |
| 3 | **CPU** | ~10-20 min | Download/extract Chapman |
| 4 | **CPU** | ~5-10 min | Build Chapman index + save tar to Drive |
| 5 | **GPU** | 1 min | Switch to GPU + mount Drive + copy scripts |
| 6 | **GPU** | ~20-30 min | Restore Chapman + PTB-XL + Challenge data + v2 model |
| 7 | **GPU** | ~2-4 hrs | Run V3 training (26 classes) |
| 8 | **GPU** | 1 min | Save model to Drive |

**Before running Cells 5-8, upload these files to `MyDrive/EKG_models/`:**
- Scripts: `multilabel_v3.py`, `multilabel_classifier.py`, `cnn_classifier.py`, `dataset_chapman.py`, `dataset_challenge.py`
- Models: `ecg_multilabel_v2.pt` (v2 checkpoint for transfer learning)
- Data tars: `chapman.tar`, `ptbxl.tar`, `challenge2021.tar`

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Cell 1  â”€â”€  CPU runtime  |  Mount Drive + install dependencies
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import time
t0 = time.time()
print('=== Cell 1 started: Mount Drive + install deps ===')

from google.colab import drive
print('Mounting Drive...')
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

import subprocess, sys
print('Installing wfdb, scipy, scikit-learn...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Packages installed.')

import torch
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  (expected False on CPU runtime)')
print(f'=== Cell 1 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Cell 2  â”€â”€  CPU runtime  |  Copy scripts from Drive to /content/
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import time, os, shutil
t0 = time.time()
print('=== Cell 2 started: Copy scripts from Drive ===')

SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
NEEDED = ['multilabel_merged.py', 'multilabel_classifier.py',
          'cnn_classifier.py', 'dataset_chapman.py']

os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)

if missing:
    raise FileNotFoundError(
        f'Missing scripts on Drive: {missing}\n'
        f'Upload them to {SCRIPTS_DIR}/ first.'
    )
print(f'=== Cell 2 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Cell 3  â”€â”€  CPU runtime  |  Download + extract Chapman ZIP (~2.3 GB)
# Expected: 10-20 min download + 5 min extract
# Safe to re-run: skips if already extracted
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import time, os, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=== Cell 3 started: Chapman download + extract ===')

CHAPMAN_DIR = '/content/ekg_datasets/chapman'
LOCAL_ZIP   = '/content/chapman.zip'
DRIVE_ZIP   = '/content/drive/MyDrive/EKG_models/chapman.zip'
ZIP_URL     = ('https://physionet.org/static/published-projects/ecg-arrhythmia/'
               'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0.zip')

os.makedirs(CHAPMAN_DIR, exist_ok=True)
n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))

if n_existing >= 44000:
    print(f'Chapman already extracted: {n_existing} .mat files - skipping')
else:
    if os.path.exists(DRIVE_ZIP):
        size_gb = os.path.getsize(DRIVE_ZIP) / 1e9
        print(f'[1/3] Copying ZIP from Drive ({size_gb:.1f} GB)...')
        shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'      Done ({time.time()-t0:.0f}s)')
    else:
        print(f'[1/3] Downloading Chapman ZIP from PhysioNet (~2.3 GB)...')
        result = subprocess.run(['wget', '-c', '-q', '--show-progress',
                                 '-O', LOCAL_ZIP, ZIP_URL], check=False)
        if result.returncode != 0 or not os.path.exists(LOCAL_ZIP):
            raise RuntimeError('Download failed. Upload chapman.zip to EKG_models/ manually.')
        size_gb = os.path.getsize(LOCAL_ZIP) / 1e9
        print(f'      Downloaded {size_gb:.1f} GB ({time.time()-t0:.0f}s)')

    print(f'[2/3] Extracting ZIP (45K files, ~5 min)...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', '/content/chapman_extract'], check=True)
    print(f'      Extracted ({time.time()-t0:.0f}s)')

    print(f'[3/3] Moving files to {CHAPMAN_DIR}...')
    extract_base = Path('/content/chapman_extract')
    wfdb_matches = list(extract_base.rglob('WFDBRecords'))
    if wfdb_matches:
        dest = Path(CHAPMAN_DIR) / 'WFDBRecords'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(wfdb_matches[0]), str(dest))
    else:
        for item in extract_base.iterdir():
            shutil.move(str(item), CHAPMAN_DIR)
    shutil.rmtree('/content/chapman_extract', ignore_errors=True)
    os.remove(LOCAL_ZIP)

n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
print(f'Done: {n_after}/45152 .mat files')
if n_after < 44000:
    print(f'WARNING: only {n_after} files â€” re-run to retry')
print(f'=== Cell 3 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Cell 4  â”€â”€  CPU runtime  |  Build Chapman index + save to Drive
# Expected: ~5-10 min
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import time, os, shutil
t0 = time.time()
print('=== Cell 4 started: Build Chapman index ===')
os.chdir('/content')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
INDEX_PATH   = '/content/ekg_datasets/chapman_index.csv'

if os.path.exists(INDEX_PATH):
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index already exists: {n} records - skipping build')
else:
    print('Building Chapman index (~5-10 min)...')
    import dataset_chapman
    dataset_chapman.build_chapman_index('/content/ekg_datasets/chapman', INDEX_PATH)
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index built: {n} records ({time.time()-t0:.0f}s)')

os.makedirs(DRIVE_MODELS, exist_ok=True)
drive_index = f'{DRIVE_MODELS}/chapman_index.csv'
shutil.copy(INDEX_PATH, drive_index)
print(f'Index saved to Drive: {drive_index}')

print(f'=== Cell 4 done ({time.time()-t0:.0f}s) ===')
print('>>> NEXT: Runtime > Change runtime type > T4 GPU > Save, then run Cells 5-8')

In [ ]:
# -----------------------------------------------------------------------------
# Cell 5  --  GPU runtime  |  Verify GPU + mount Drive + copy scripts
# Run this FIRST after switching to GPU runtime
# -----------------------------------------------------------------------------
NOTEBOOK_VERSION = "v3"
import time, subprocess, sys
t0 = time.time()
print('=== Cell 5 started: GPU check + setup (notebook ' + NOTEBOOK_VERSION + ') ===')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU > Save')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
print('Mounting Drive...')
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Packages installed.')

import os, shutil
SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

NEEDED = ['multilabel_v3.py', 'multilabel_classifier.py',
          'cnn_classifier.py', 'dataset_chapman.py', 'dataset_challenge.py']
missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)

if missing:
    raise FileNotFoundError(
        'Missing scripts on Drive: ' + str(missing) + '. Upload them to ' + SCRIPTS_DIR + '/ first.'
    )
print(f'=== Cell 5 done ({time.time()-t0:.0f}s) ===')


In [ ]:
# -----------------------------------------------------------------------------
# Cell 6  --  GPU runtime  |  Restore all data to local SSD
# Chapman:   tar extract from Drive  ~5 min
# PTB-XL:    tar extract from Drive  ~8 min
# Challenge: tar extract from Drive  ~10 min
# v2 model:  copy from Drive for transfer learning
# -----------------------------------------------------------------------------
import time, os, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=== Cell 6 started: Restore data to local SSD ===')
os.chdir('/content')

DRIVE_MODELS  = '/content/drive/MyDrive/EKG_models'
CHAPMAN_DIR   = '/content/ekg_datasets/chapman'
CHAPMAN_INDEX = '/content/ekg_datasets/chapman_index.csv'
PTBXL_LOCAL   = '/content/ekg_datasets/ptbxl'
os.makedirs(CHAPMAN_DIR, exist_ok=True)
os.makedirs(PTBXL_LOCAL, exist_ok=True)

# -- Step 1: Restore Chapman --------------------------------------------------
n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
if n_existing >= 44000:
    print(f'[1/5] Chapman already present: {n_existing} .mat files - skipping')
else:
    DRIVE_TAR = f'{DRIVE_MODELS}/chapman.tar'
    if not os.path.exists(DRIVE_TAR):
        raise FileNotFoundError('chapman.tar not found: ' + DRIVE_TAR + '. Run CPU Cells 1-4 first.')
    size_gb = os.path.getsize(DRIVE_TAR) / 1e9
    print(f'[1/5] Extracting chapman.tar ({size_gb:.1f} GB)...')
    subprocess.run(['tar', 'xf', DRIVE_TAR, '-C', '/content'], check=True)
    n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
    print(f'      Chapman restored: {n_after} .mat files ({time.time()-t0:.0f}s)')

# -- Step 2: Restore Chapman index --------------------------------------------
if os.path.exists(CHAPMAN_INDEX):
    import pandas as pd
    print(f'[2/5] Chapman index present: {len(pd.read_csv(CHAPMAN_INDEX))} records')
else:
    DRIVE_INDEX = f'{DRIVE_MODELS}/chapman_index.csv'
    if os.path.exists(DRIVE_INDEX):
        shutil.copy(DRIVE_INDEX, CHAPMAN_INDEX)
        import pandas as pd
        print(f'[2/5] Chapman index restored: {len(pd.read_csv(CHAPMAN_INDEX))} records ({time.time()-t0:.0f}s)')
    else:
        print('[2/5] Building Chapman index (~5-10 min)...')
        import dataset_chapman
        dataset_chapman.build_chapman_index(CHAPMAN_DIR, CHAPMAN_INDEX)
        import pandas as pd
        print(f'      Index built: {len(pd.read_csv(CHAPMAN_INDEX))} records ({time.time()-t0:.0f}s)')

# -- Step 3: Restore PTB-XL ---------------------------------------------------
n_local = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
if n_local >= 18000:
    print(f'[3/5] PTB-XL already on SSD: {n_local} .dat files - skipping')
else:
    DRIVE_TAR = f'{DRIVE_MODELS}/ptbxl.tar'
    if not os.path.exists(DRIVE_TAR):
        raise FileNotFoundError(
            'ptbxl.tar not found: ' + str(DRIVE_TAR) + '. Upload ptbxl.tar to EKG_models/ on Drive.'
        )
    size_gb = os.path.getsize(DRIVE_TAR) / 1e9
    print(f'[3/5] Extracting ptbxl.tar ({size_gb:.1f} GB, ~8 min)...')
    subprocess.run(['tar', 'xf', DRIVE_TAR, '-C', '/content/ekg_datasets'], check=True)
    n_local = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'      PTB-XL restored: {n_local} .dat files ({time.time()-t0:.0f}s)')

if len(list(Path(PTBXL_LOCAL).rglob('*.dat'))) < 18000:
    raise RuntimeError('PTB-XL incomplete. Check ptbxl.tar on Drive.')

# -- Step 4: Restore v2 model (for transfer learning) -------------------------
V2_MODEL = '/content/models/ecg_multilabel_v2.pt'
if os.path.exists(V2_MODEL):
    print(f'[4/5] v2 model already present: {V2_MODEL}')
else:
    DRIVE_V2 = f'{DRIVE_MODELS}/ecg_multilabel_v2.pt'
    if not os.path.exists(DRIVE_V2):
        raise FileNotFoundError(
            'v2 model not found: ' + DRIVE_V2 + '. Upload ecg_multilabel_v2.pt to EKG_models/ on Drive.'
        )
    shutil.copy(DRIVE_V2, V2_MODEL)
    size_mb = os.path.getsize(V2_MODEL) / 1e6
    print(f'[4/5] v2 model restored ({size_mb:.1f} MB)')

# -- Step 5: Restore challenge2021 data ---------------------------------------
CHALLENGE_DIR = '/content/ekg_datasets/challenge2021'
os.makedirs(CHALLENGE_DIR, exist_ok=True)
n_challenge = len(list(Path(CHALLENGE_DIR).rglob('*.mat')))
if n_challenge >= 50000:
    print(f'[5/5] Challenge data already present: {n_challenge} .mat files - skipping')
else:
    DRIVE_CH_TAR = f'{DRIVE_MODELS}/challenge2021.tar'
    if os.path.exists(DRIVE_CH_TAR):
        size_gb = os.path.getsize(DRIVE_CH_TAR) / 1e9
        print(f'[5/5] Extracting challenge2021.tar ({size_gb:.1f} GB)...')
        subprocess.run(['tar', 'xf', DRIVE_CH_TAR, '-C', '/content/ekg_datasets'], check=True)
        n_after = len(list(Path(CHALLENGE_DIR).rglob('*.mat')))
        print(f'      Challenge data restored: {n_after} .mat files ({time.time()-t0:.0f}s)')
    else:
        print('[5/5] WARNING: challenge2021.tar not found on Drive.')
        print('      Skipping challenge data -- training on PTB-XL + Chapman only.')
        print('      To include: create challenge2021.tar locally and upload to EKG_models/.')

print(f'=== Cell 6 done ({time.time()-t0:.0f}s) -- data ready ===')


In [ ]:
# -----------------------------------------------------------------------------
# Cell 7  --  GPU runtime  |  Train V3 26-class model
# Expected: ~2-4 hrs on T4  |  Streams output line-by-line in real time
# Transfer learning from ecg_multilabel_v2.pt (14 -> 26 classes)
# -----------------------------------------------------------------------------
import time, os, sys, subprocess
t0 = time.time()
print('=== Cell 7 started: V3 Training (26 classes) ===')
os.chdir('/content')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Run Cell 5 first.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Starting multilabel_v3.py -- output streams below:', flush=True)
print('-' * 60, flush=True)

proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_v3.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    print(f'Training exited with code {proc.returncode}')
else:
    model_path = '/content/models/ecg_multilabel_v3.pt'
    if os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / 1e6
        print(f'Model saved: {model_path} ({size_mb:.1f} MB)')
    else:
        print('WARNING: Model file not found')
print(f'=== Cell 7 done ({time.time()-t0:.0f}s) ===')


In [ ]:
# -----------------------------------------------------------------------------
# Cell 8  --  GPU runtime  |  Save trained V3 model to Drive
# -----------------------------------------------------------------------------
import time, os, shutil
t0 = time.time()
print('=== Cell 8 started: Save V3 model to Drive ===')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

src = '/content/models/ecg_multilabel_v3.pt'
dst = f'{DRIVE_MODELS}/ecg_multilabel_v3.pt'

if os.path.exists(src):
    print(f'Copying {src} to Drive...')
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'Saved: {dst} ({size_mb:.1f} MB)')
else:
    print(f'WARNING: {src} not found - training may have failed')

print(f'=== Cell 8 done ({time.time()-t0:.0f}s) ===')
print('Download ecg_multilabel_v3.pt from Drive and place in models/ locally.')
